# 04 — Build Reporting Marts

## Banking Reporting Platform

This notebook transforms the validated staging layer into the reporting-ready dimensional model used by Metabase.

Pipeline position:

```text
raw
 ↓
staging
 ↓
marts
```

The mart layer contains:

```text
marts.dim_customer
marts.dim_account
marts.dim_channel
marts.dim_date
marts.bridge_account_customer
marts.fact_transactions
```

The model implements the design agreed in the data-modelling documents:

- one transaction fact table
- customer, account, channel, and date dimensions
- a Customer–Account bridge for the many-to-many relationship
- equal allocation weights for joint accounts
- **SCD Type 1** handling for `dim_customer` and `dim_account`

### Load strategy

The mart build uses two different patterns:

**Dimensions**
- insert new business keys
- update existing business keys in place
- preserve the existing surrogate key
- do not create historical versions

That is the project's SCD Type 1 behaviour.

**Bridge and fact**
- full refresh from the current validated staging snapshot
- rebuild after dimensions have been upserted

This keeps the project simple while still demonstrating realistic dimension maintenance.


## 1. Imports and Database Configuration


In [1]:
from pathlib import Path

import pandas as pd
import psycopg
from psycopg import sql
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected a data/raw directory "
        "in the current directory or one of its parents."
    )


def read_env_file(path: Path) -> dict[str, str]:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} was not found. Create .env from .env.example before continuing."
        )

    values = {}

    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()

        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")

    return values


PROJECT_ROOT = find_project_root(Path.cwd())
ENV_PATH = PROJECT_ROOT / ".env"

env = read_env_file(ENV_PATH)

DB_CONFIG = {
    "host": "localhost",
    "port": int(env["POSTGRES_PORT"]),
    "dbname": env["POSTGRES_DB"],
    "user": env["POSTGRES_USER"],
    "password": env["POSTGRES_PASSWORD"],
}


def get_connection():
    return psycopg.connect(**DB_CONFIG)


def query_dataframe(query: str, params=None) -> pd.DataFrame:
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)
            columns = [description.name for description in cur.description]
            rows = cur.fetchall()

    return pd.DataFrame(rows, columns=columns)


print(f"Project root: {PROJECT_ROOT}")
print(
    "PostgreSQL target:",
    f"{DB_CONFIG['user']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}",
)


Project root: C:\Users\Admin\Projects\banking-reporting-platform
PostgreSQL target: banking_admin@localhost:5432/analytics


## 2. Confirm the Staging Layer Is Ready

Notebook 03 must have completed successfully before building the marts.

The mart build expects validated rows in:

```text
staging.customers
staging.accounts
staging.customer_accounts
staging.channels
staging.transactions
```


In [2]:
STAGING_TABLES = [
    "customers",
    "accounts",
    "customer_accounts",
    "channels",
    "transactions",
]

staging_counts = []

with get_connection() as conn:
    with conn.cursor() as cur:
        for table_name in STAGING_TABLES:
            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM staging.{}").format(
                    sql.Identifier(table_name)
                )
            )

            staging_counts.append(
                {
                    "staging_table": f"staging.{table_name}",
                    "rows": cur.fetchone()[0],
                }
            )

staging_counts = pd.DataFrame(staging_counts)
display(staging_counts)

required_non_empty = {
    "staging.customers",
    "staging.accounts",
    "staging.customer_accounts",
    "staging.channels",
    "staging.transactions",
}

empty_required = staging_counts.loc[
    staging_counts["staging_table"].isin(required_non_empty)
    & staging_counts["rows"].eq(0),
    "staging_table",
].tolist()

if empty_required:
    raise RuntimeError(
        f"Required staging tables are empty: {empty_required}. "
        "Run 03_transform_staging.ipynb first."
    )


,staging_table,rows
0,staging.customers,1000
1,staging.accounts,1250
2,staging.customer_accounts,1350
3,staging.channels,3
4,staging.transactions,50000


## 3. Create the Mart Tables

These structures follow the approved dimensional and physical models.

`dim_customer` and `dim_account` use surrogate keys while retaining their source business keys.

The bridge resolves the many-to-many Customer–Account relationship.


In [3]:
MART_DDL = '''
CREATE SCHEMA IF NOT EXISTS marts;

CREATE TABLE IF NOT EXISTS marts.dim_customer (
    customer_key BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    customer_id VARCHAR(20) UNIQUE NOT NULL,
    customer_since_date DATE NOT NULL,
    customer_status VARCHAR(10) NOT NULL
        CHECK (customer_status IN ('ACTIVE', 'INACTIVE'))
);

CREATE TABLE IF NOT EXISTS marts.dim_account (
    account_key BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    account_id VARCHAR(20) UNIQUE NOT NULL,
    account_type VARCHAR(20) NOT NULL
        CHECK (account_type IN ('TRANSACTION', 'SAVINGS')),
    account_status VARCHAR(10) NOT NULL
        CHECK (account_status IN ('ACTIVE', 'DORMANT', 'CLOSED')),
    opened_date DATE NOT NULL,
    closed_date DATE,
    CHECK (closed_date IS NULL OR closed_date >= opened_date)
);

CREATE TABLE IF NOT EXISTS marts.dim_channel (
    channel_key SMALLINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    channel_code VARCHAR(10) UNIQUE NOT NULL,
    channel_name VARCHAR(50) NOT NULL
);

CREATE TABLE IF NOT EXISTS marts.dim_date (
    date_key INTEGER PRIMARY KEY,
    full_date DATE UNIQUE NOT NULL,
    day_name VARCHAR(10) NOT NULL,
    month_number SMALLINT NOT NULL
        CHECK (month_number BETWEEN 1 AND 12),
    month_name VARCHAR(10) NOT NULL,
    quarter_number SMALLINT NOT NULL
        CHECK (quarter_number BETWEEN 1 AND 4),
    year SMALLINT NOT NULL
);

CREATE TABLE IF NOT EXISTS marts.bridge_account_customer (
    account_key BIGINT NOT NULL
        REFERENCES marts.dim_account(account_key),
    customer_key BIGINT NOT NULL
        REFERENCES marts.dim_customer(customer_key),
    holder_role VARCHAR(10) NOT NULL
        CHECK (holder_role IN ('PRIMARY', 'JOINT')),
    allocation_weight NUMERIC(8, 6) NOT NULL
        CHECK (allocation_weight > 0 AND allocation_weight <= 1),
    PRIMARY KEY (account_key, customer_key)
);

CREATE TABLE IF NOT EXISTS marts.fact_transactions (
    transaction_key BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    transaction_id VARCHAR(30) UNIQUE NOT NULL,
    account_key BIGINT NOT NULL
        REFERENCES marts.dim_account(account_key),
    channel_key SMALLINT NOT NULL
        REFERENCES marts.dim_channel(channel_key),
    date_key INTEGER NOT NULL
        REFERENCES marts.dim_date(date_key),
    transaction_timestamp TIMESTAMP NOT NULL,
    transaction_type VARCHAR(20) NOT NULL
        CHECK (transaction_type IN ('PURCHASE', 'WITHDRAWAL', 'DEPOSIT', 'TRANSFER')),
    amount NUMERIC(18, 2) NOT NULL
        CHECK (amount > 0),
    currency_code CHAR(3) NOT NULL
        CHECK (currency_code = 'ZAR'),
    status VARCHAR(15) NOT NULL
        CHECK (status IN ('SUCCESSFUL', 'FAILED', 'REVERSED'))
);

CREATE INDEX IF NOT EXISTS idx_fact_transactions_account_key
    ON marts.fact_transactions(account_key);

CREATE INDEX IF NOT EXISTS idx_fact_transactions_channel_key
    ON marts.fact_transactions(channel_key);

CREATE INDEX IF NOT EXISTS idx_fact_transactions_date_key
    ON marts.fact_transactions(date_key);

CREATE INDEX IF NOT EXISTS idx_bridge_account_customer_customer_key
    ON marts.bridge_account_customer(customer_key);
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(MART_DDL)

print("Mart schema and tables are ready.")


Mart schema and tables are ready.


## 4. Capture Dimension State Before the Load

This lets us see what the SCD Type 1 load actually did.

On the first run, all dimension records will be inserts.

On a later run, existing business keys can be updated in place while keeping the same surrogate keys.


In [4]:
dimension_state_before = {
    "dim_customer": query_dataframe(
        '''
        SELECT
            customer_key,
            customer_id,
            customer_since_date,
            customer_status
        FROM marts.dim_customer;
        '''
    ),
    "dim_account": query_dataframe(
        '''
        SELECT
            account_key,
            account_id,
            account_type,
            account_status,
            opened_date,
            closed_date
        FROM marts.dim_account;
        '''
    ),
}

print(
    "Existing dim_customer rows:",
    len(dimension_state_before["dim_customer"]),
)
print(
    "Existing dim_account rows:",
    len(dimension_state_before["dim_account"]),
)


Existing dim_customer rows: 0
Existing dim_account rows: 0


## 5. Upsert `dim_customer` — SCD Type 1

Type 1 means:

```text
new customer_id
→ INSERT

existing customer_id
→ UPDATE descriptive attributes in the same row
→ retain customer_key
```

No historical version row is created.


In [5]:
CUSTOMER_UPSERT = '''
INSERT INTO marts.dim_customer (
    customer_id,
    customer_since_date,
    customer_status
)
SELECT
    customer_id,
    customer_since_date,
    customer_status
FROM staging.customers
ON CONFLICT (customer_id)
DO UPDATE SET
    customer_since_date = EXCLUDED.customer_since_date,
    customer_status = EXCLUDED.customer_status;
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(CUSTOMER_UPSERT)
        affected_customer_rows = cur.rowcount

print(f"Customer dimension rows inserted/updated: {affected_customer_rows:,}")


Customer dimension rows inserted/updated: 1,000


## 6. Upsert `dim_account` — SCD Type 1

The same Type 1 rule applies to accounts:

```text
new account_id
→ INSERT

existing account_id
→ UPDATE current attributes
→ retain account_key
```


In [6]:
ACCOUNT_UPSERT = '''
INSERT INTO marts.dim_account (
    account_id,
    account_type,
    account_status,
    opened_date,
    closed_date
)
SELECT
    account_id,
    account_type,
    account_status,
    opened_date,
    closed_date
FROM staging.accounts
ON CONFLICT (account_id)
DO UPDATE SET
    account_type = EXCLUDED.account_type,
    account_status = EXCLUDED.account_status,
    opened_date = EXCLUDED.opened_date,
    closed_date = EXCLUDED.closed_date;
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(ACCOUNT_UPSERT)
        affected_account_rows = cur.rowcount

print(f"Account dimension rows inserted/updated: {affected_account_rows:,}")


Account dimension rows inserted/updated: 1,250


## 7. Upsert `dim_channel`

The channel dimension is small reference data.

We keep the same upsert pattern so the notebook remains rerunnable.


In [7]:
CHANNEL_UPSERT = '''
INSERT INTO marts.dim_channel (
    channel_code,
    channel_name
)
SELECT
    channel_code,
    channel_name
FROM staging.channels
ON CONFLICT (channel_code)
DO UPDATE SET
    channel_name = EXCLUDED.channel_name;
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(CHANNEL_UPSERT)
        affected_channel_rows = cur.rowcount

print(f"Channel dimension rows inserted/updated: {affected_channel_rows:,}")


Channel dimension rows inserted/updated: 3


## 8. Build `dim_date`

The date dimension is generated from the minimum to maximum transaction date in staging.

The integer key follows:

```text
YYYYMMDD
```

Example:

```text
20260905
```


In [8]:
DATE_DIMENSION_INSERT = '''
WITH date_bounds AS (
    SELECT
        MIN(transaction_timestamp::date) AS min_date,
        MAX(transaction_timestamp::date) AS max_date
    FROM staging.transactions
),
calendar AS (
    SELECT
        generate_series(
            min_date,
            max_date,
            INTERVAL '1 day'
        )::date AS full_date
    FROM date_bounds
)
INSERT INTO marts.dim_date (
    date_key,
    full_date,
    day_name,
    month_number,
    month_name,
    quarter_number,
    year
)
SELECT
    TO_CHAR(full_date, 'YYYYMMDD')::INTEGER AS date_key,
    full_date,
    TO_CHAR(full_date, 'FMDay') AS day_name,
    EXTRACT(MONTH FROM full_date)::SMALLINT AS month_number,
    TO_CHAR(full_date, 'FMMonth') AS month_name,
    EXTRACT(QUARTER FROM full_date)::SMALLINT AS quarter_number,
    EXTRACT(YEAR FROM full_date)::SMALLINT AS year
FROM calendar
ON CONFLICT (date_key)
DO UPDATE SET
    full_date = EXCLUDED.full_date,
    day_name = EXCLUDED.day_name,
    month_number = EXCLUDED.month_number,
    month_name = EXCLUDED.month_name,
    quarter_number = EXCLUDED.quarter_number,
    year = EXCLUDED.year;
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(DATE_DIMENSION_INSERT)
        affected_date_rows = cur.rowcount

print(f"Date dimension rows inserted/updated: {affected_date_rows:,}")


Date dimension rows inserted/updated: 3,140


## 9. Verify SCD Type 1 Surrogate-Key Stability

For business keys that existed before this run, their surrogate keys must remain unchanged.

This is one of the key differences between the current Type 1 design and an SCD Type 2 design.


In [9]:
dimension_state_after = {
    "dim_customer": query_dataframe(
        '''
        SELECT
            customer_key,
            customer_id,
            customer_since_date,
            customer_status
        FROM marts.dim_customer;
        '''
    ),
    "dim_account": query_dataframe(
        '''
        SELECT
            account_key,
            account_id,
            account_type,
            account_status,
            opened_date,
            closed_date
        FROM marts.dim_account;
        '''
    ),
}


def count_changed_keys(before, after, business_key, surrogate_key):
    if before.empty:
        return 0

    comparison = before[
        [business_key, surrogate_key]
    ].merge(
        after[[business_key, surrogate_key]],
        on=business_key,
        how="inner",
        suffixes=("_before", "_after"),
    )

    return int(
        (
            comparison[f"{surrogate_key}_before"]
            != comparison[f"{surrogate_key}_after"]
        ).sum()
    )


customer_key_changes = count_changed_keys(
    dimension_state_before["dim_customer"],
    dimension_state_after["dim_customer"],
    "customer_id",
    "customer_key",
)

account_key_changes = count_changed_keys(
    dimension_state_before["dim_account"],
    dimension_state_after["dim_account"],
    "account_id",
    "account_key",
)

scd_key_check = pd.DataFrame(
    [
        {
            "dimension": "marts.dim_customer",
            "existing_surrogate_keys_changed": customer_key_changes,
            "passed": customer_key_changes == 0,
        },
        {
            "dimension": "marts.dim_account",
            "existing_surrogate_keys_changed": account_key_changes,
            "passed": account_key_changes == 0,
        },
    ]
)

display(scd_key_check)

if not scd_key_check["passed"].all():
    raise RuntimeError("SCD Type 1 surrogate-key stability check failed.")


,dimension,existing_surrogate_keys_changed,passed
0,marts.dim_customer,0,True
1,marts.dim_account,0,True


## 10. Rebuild the Customer–Account Bridge

The bridge represents the **current** account-holder relationships from staging.

Because staging is a current full snapshot, the bridge is fully rebuilt on each mart run.


In [10]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE marts.bridge_account_customer;")

print("Existing bridge rows removed.")


Existing bridge rows removed.


## 11. Load the Bridge with Allocation Weights

The allocation weight is:

```text
1 / number_of_holders_on_the_account
```

Examples:

```text
1 holder  → 1.000000
2 holders → 0.500000
3 holders → 0.333333
```

This prevents customer-level allocated transaction values from double-counting joint-account activity.


In [11]:
BRIDGE_INSERT = '''
WITH holder_counts AS (
    SELECT
        account_id,
        COUNT(*) AS holder_count
    FROM staging.customer_accounts
    GROUP BY account_id
)
INSERT INTO marts.bridge_account_customer (
    account_key,
    customer_key,
    holder_role,
    allocation_weight
)
SELECT
    da.account_key,
    dc.customer_key,
    ca.holder_role,
    ROUND(
        1.0::NUMERIC / hc.holder_count,
        6
    )::NUMERIC(8, 6) AS allocation_weight
FROM staging.customer_accounts ca
JOIN holder_counts hc
    ON hc.account_id = ca.account_id
JOIN marts.dim_account da
    ON da.account_id = ca.account_id
JOIN marts.dim_customer dc
    ON dc.customer_id = ca.customer_id;
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(BRIDGE_INSERT)
        bridge_rows_loaded = cur.rowcount

print(f"Bridge rows loaded: {bridge_rows_loaded:,}")


Bridge rows loaded: 1,350


## 12. Validate Bridge Allocation

For each account, the sum of allocation weights should equal approximately `1.0`.


In [12]:
bridge_allocation_check = query_dataframe(
    '''
    SELECT
        da.account_id,
        COUNT(*) AS holders,
        SUM(bac.allocation_weight) AS allocation_total
    FROM marts.bridge_account_customer bac
    JOIN marts.dim_account da
        ON da.account_key = bac.account_key
    GROUP BY da.account_id
    HAVING ABS(SUM(bac.allocation_weight) - 1.0) > 0.000001
    ORDER BY da.account_id;
    '''
)

display(bridge_allocation_check)

if not bridge_allocation_check.empty:
    raise RuntimeError(
        "One or more accounts have allocation weights that do not sum to 1."
    )

print("All bridge allocation weights reconcile to 1.0 per account.")


,account_id,holders,allocation_total


All bridge allocation weights reconcile to 1.0 per account.


## 13. Full-Refresh the Transaction Fact

The current source is a complete transaction extract, so the fact table is rebuilt from the current staging snapshot.

Dimensions are **not** truncated because their surrogate keys must remain stable for SCD Type 1.


In [13]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(
            '''
            TRUNCATE TABLE marts.fact_transactions
            RESTART IDENTITY;
            '''
        )

print("Existing transaction fact rows removed.")


Existing transaction fact rows removed.


## 14. Load `fact_transactions`

The staging business keys are translated into surrogate dimension keys during the load.

Fact grain:

> **One row per validated financial transaction processed against one account.**


In [14]:
FACT_INSERT = '''
INSERT INTO marts.fact_transactions (
    transaction_id,
    account_key,
    channel_key,
    date_key,
    transaction_timestamp,
    transaction_type,
    amount,
    currency_code,
    status
)
SELECT
    t.transaction_id,
    da.account_key,
    dc.channel_key,
    dd.date_key,
    t.transaction_timestamp,
    t.transaction_type,
    t.amount,
    t.currency_code,
    t.status
FROM staging.transactions t
JOIN marts.dim_account da
    ON da.account_id = t.account_id
JOIN marts.dim_channel dc
    ON dc.channel_code = t.channel_code
JOIN marts.dim_date dd
    ON dd.full_date = t.transaction_timestamp::date;
'''


with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(FACT_INSERT)
        fact_rows_loaded = cur.rowcount

print(f"Fact rows loaded: {fact_rows_loaded:,}")


Fact rows loaded: 50,000


## 15. Reconcile Staging to Marts

The main controls are:

```text
staging customers          ↔ current source customer dimension coverage
staging accounts           ↔ current source account dimension coverage
staging customer_accounts  = bridge rows
staging transactions       = fact rows
```

The customer and account dimensions may contain older business keys from previous full extracts because SCD Type 1 preserves existing dimension rows. Therefore we check **coverage of the current staging keys**, not strict total dimension row equality.


In [15]:
reconciliation = query_dataframe(
    '''
    SELECT
        'customers represented in dim_customer' AS check_name,
        (SELECT COUNT(*) FROM staging.customers) AS source_rows,
        (
            SELECT COUNT(*)
            FROM staging.customers s
            JOIN marts.dim_customer d
                ON d.customer_id = s.customer_id
        ) AS mart_rows

    UNION ALL

    SELECT
        'accounts represented in dim_account',
        (SELECT COUNT(*) FROM staging.accounts),
        (
            SELECT COUNT(*)
            FROM staging.accounts s
            JOIN marts.dim_account d
                ON d.account_id = s.account_id
        )

    UNION ALL

    SELECT
        'customer-account relationships in bridge',
        (SELECT COUNT(*) FROM staging.customer_accounts),
        (SELECT COUNT(*) FROM marts.bridge_account_customer)

    UNION ALL

    SELECT
        'transactions in fact',
        (SELECT COUNT(*) FROM staging.transactions),
        (SELECT COUNT(*) FROM marts.fact_transactions);
    '''
)

reconciliation["difference"] = (
    reconciliation["mart_rows"] - reconciliation["source_rows"]
)
reconciliation["reconciled"] = reconciliation["difference"].eq(0)

display(reconciliation)

if not reconciliation["reconciled"].all():
    raise RuntimeError("Staging-to-mart reconciliation failed.")

print("All current staging records reconcile to the mart layer.")


,check_name,source_rows,mart_rows,difference,reconciled
0,customers represented in dim_customer,1000,1000,0,True
1,accounts represented in dim_account,1250,1250,0,True
2,customer-account relationships in bridge,1350,1350,0,True
3,transactions in fact,50000,50000,0,True


All current staging records reconcile to the mart layer.


## 16. Validate Fact Foreign Keys and Grain

These checks confirm:

- every fact row has valid dimension keys
- `transaction_id` remains unique
- the fact grain has not been duplicated


In [16]:
fact_quality_checks = query_dataframe(
    '''
    SELECT
        'duplicate transaction_id' AS check_name,
        COUNT(*) AS failed_rows
    FROM (
        SELECT transaction_id
        FROM marts.fact_transactions
        GROUP BY transaction_id
        HAVING COUNT(*) > 1
    ) duplicates

    UNION ALL

    SELECT
        'missing account dimension key',
        COUNT(*)
    FROM marts.fact_transactions f
    LEFT JOIN marts.dim_account d
        ON d.account_key = f.account_key
    WHERE d.account_key IS NULL

    UNION ALL

    SELECT
        'missing channel dimension key',
        COUNT(*)
    FROM marts.fact_transactions f
    LEFT JOIN marts.dim_channel d
        ON d.channel_key = f.channel_key
    WHERE d.channel_key IS NULL

    UNION ALL

    SELECT
        'missing date dimension key',
        COUNT(*)
    FROM marts.fact_transactions f
    LEFT JOIN marts.dim_date d
        ON d.date_key = f.date_key
    WHERE d.date_key IS NULL;
    '''
)

fact_quality_checks["passed"] = fact_quality_checks["failed_rows"].eq(0)

display(fact_quality_checks)

if not fact_quality_checks["passed"].all():
    raise RuntimeError("One or more fact-table quality checks failed.")

print("Fact grain and dimension-key checks passed.")


,check_name,failed_rows,passed
0,missing channel dimension key,0,True
1,missing account dimension key,0,True
2,missing date dimension key,0,True
3,duplicate transaction_id,0,True


Fact grain and dimension-key checks passed.


## 17. Inspect the Final Mart Tables


In [17]:
mart_counts = []

with get_connection() as conn:
    with conn.cursor() as cur:
        for table_name in [
            "dim_customer",
            "dim_account",
            "dim_channel",
            "dim_date",
            "bridge_account_customer",
            "fact_transactions",
        ]:
            cur.execute(
                sql.SQL("SELECT COUNT(*) FROM marts.{}").format(
                    sql.Identifier(table_name)
                )
            )

            mart_counts.append(
                {
                    "mart_table": f"marts.{table_name}",
                    "rows": cur.fetchone()[0],
                }
            )

display(pd.DataFrame(mart_counts))


,mart_table,rows
0,marts.dim_customer,1000
1,marts.dim_account,1250
2,marts.dim_channel,3
3,marts.dim_date,3140
4,marts.bridge_account_customer,1350
5,marts.fact_transactions,50000


## 18. Inspect Sample Fact and Dimension Data

This gives a human-readable view of how the surrogate-key model comes together.


In [18]:
sample_transactions = query_dataframe(
    '''
    SELECT
        f.transaction_id,
        f.transaction_timestamp,
        f.transaction_type,
        f.amount,
        f.status,
        a.account_id,
        a.account_type,
        c.channel_name,
        d.full_date
    FROM marts.fact_transactions f
    JOIN marts.dim_account a
        ON a.account_key = f.account_key
    JOIN marts.dim_channel c
        ON c.channel_key = f.channel_key
    JOIN marts.dim_date d
        ON d.date_key = f.date_key
    ORDER BY f.transaction_timestamp
    LIMIT 10;
    '''
)

display(sample_transactions)


,transaction_id,transaction_timestamp,transaction_type,amount,status,account_id,account_type,channel_name,full_date
0,T00003707,2018-01-26 00:57:14,PURCHASE,423.13,SUCCESSFUL,A000420,SAVINGS,Card,2018-01-26
1,T00021081,2018-01-30 22:14:07,PURCHASE,304.50,SUCCESSFUL,A000554,SAVINGS,Card,2018-01-30
2,T00009177,2018-02-01 02:37:42,WITHDRAWAL,200.00,SUCCESSFUL,A000812,TRANSACTION,ATM,2018-02-01
3,T00036823,2018-02-02 21:35:52,DEPOSIT,9400.88,SUCCESSFUL,A000812,TRANSACTION,ATM,2018-02-02
4,T00040657,2018-02-05 03:58:32,TRANSFER,12920.67,SUCCESSFUL,A000420,SAVINGS,Mobile App,2018-02-05
5,T00026646,2018-02-10 00:42:27,PURCHASE,938.82,SUCCESSFUL,A000534,SAVINGS,Mobile App,2018-02-10
6,T00018729,2018-02-11 04:57:20,PURCHASE,116.57,SUCCESSFUL,A000449,TRANSACTION,Card,2018-02-11
7,T00032788,2018-02-12 02:14:06,TRANSFER,8620.38,SUCCESSFUL,A000758,SAVINGS,Mobile App,2018-02-12
8,T00015499,2018-02-14 01:49:54,WITHDRAWAL,2500.00,SUCCESSFUL,A000534,SAVINGS,ATM,2018-02-14
9,T00029436,2018-02-18 10:38:06,PURCHASE,217.75,SUCCESSFUL,A000758,SAVINGS,Card,2018-02-18


## 19. Demonstrate Joint-Account Allocation

This query shows some multi-holder accounts and confirms that allocation weights are split across their holders.


In [19]:
joint_account_sample = query_dataframe(
    '''
    SELECT
        a.account_id,
        c.customer_id,
        bac.holder_role,
        bac.allocation_weight
    FROM marts.bridge_account_customer bac
    JOIN marts.dim_account a
        ON a.account_key = bac.account_key
    JOIN marts.dim_customer c
        ON c.customer_key = bac.customer_key
    WHERE bac.account_key IN (
        SELECT account_key
        FROM marts.bridge_account_customer
        GROUP BY account_key
        HAVING COUNT(*) > 1
    )
    ORDER BY a.account_id, bac.holder_role DESC, c.customer_id
    LIMIT 20;
    '''
)

display(joint_account_sample)


,account_id,customer_id,holder_role,allocation_weight
0,A000011,C000856,PRIMARY,0.500000
1,A000011,C000678,JOINT,0.500000
2,A000038,C000417,PRIMARY,0.500000
3,A000038,C000327,JOINT,0.500000
4,A000041,C000965,PRIMARY,0.500000
5,A000041,C000863,JOINT,0.500000
6,A000050,C000018,PRIMARY,0.500000
7,A000050,C000727,JOINT,0.500000
8,A000053,C000195,PRIMARY,0.500000
9,A000053,C000225,JOINT,0.500000


## 20. Demonstrate Customer-Level Allocated Transaction Value

A direct join from transactions to all joint-account holders would duplicate transaction value.

The bridge allocation weight prevents that.

For customer-level reporting:

```text
allocated_amount = transaction amount × allocation_weight
```


In [20]:
allocated_customer_sample = query_dataframe(
    '''
    SELECT
        c.customer_id,
        ROUND(
            SUM(f.amount * bac.allocation_weight),
            2
        ) AS allocated_transaction_value
    FROM marts.fact_transactions f
    JOIN marts.bridge_account_customer bac
        ON bac.account_key = f.account_key
    JOIN marts.dim_customer c
        ON c.customer_key = bac.customer_key
    GROUP BY c.customer_id
    ORDER BY allocated_transaction_value DESC
    LIMIT 10;
    '''
)

display(allocated_customer_sample)


,customer_id,allocated_transaction_value
0,C000374,860629.56
1,C000998,790448.80
2,C000938,788668.40
3,C000669,782089.91
4,C000999,750317.39
5,C000711,729817.87
6,C000293,722967.15
7,C000843,696068.21
8,C000569,680643.47
9,C000513,639120.98


## 21. Mart Build Conclusion

The dimensional reporting model is now built.

```text
staging
   ↓
marts.dim_customer        ← SCD Type 1
marts.dim_account         ← SCD Type 1
marts.dim_channel
marts.dim_date
marts.bridge_account_customer
marts.fact_transactions
```

We have also verified:

- current staging business keys are represented in their dimensions
- existing Type 1 surrogate keys remain stable
- bridge allocation weights reconcile to `1.0` per account
- every staging transaction reaches the fact table
- fact business grain remains one row per transaction
- fact foreign keys resolve to valid dimensions
- customer-level joint-account reporting can use allocated measures safely

### Next step

`05_data_quality_checks.ipynb`

That final pipeline-validation notebook will test the completed database from end to end, including:

- row-count reconciliation across layers
- key uniqueness
- referential integrity
- business-rule validation
- mart allocation checks
- KPI sanity checks
- readiness for Metabase reporting

After that, we move to the dashboard layer.
